# Required vs. Optional Fields & Mutable Defaults

## 1. Defining Optional Fields with Defaults

By default, any field defined in a Pydantic model with a type hint is **required**. To make a field **optional**, you simply assign it a default value using standard Python assignment syntax (`=`).

```python
from pydantic import BaseModel, ValidationError

class Circle(BaseModel):
    radius: int              # Required field (no default)
    center: tuple[int, int] = (0, 0)  # Optional field (defaults to (0,0))

# We can instantiate without providing 'center'
c1 = Circle(radius=5)
print(c1) # Output: radius=5 center=(0, 0)

```

## 2. The Default Validation "Loophole"

By default, Pydantic **does not validate the default values** you write into your model schema. Pydantic assumes that if you (the developer) hardcode a default, you know what you are doing.

```python
class BrokenCircle(BaseModel):
    radius: int
    # BAD DEFAULT: Type hint is a tuple, but default is a string
    center: tuple[int, int] = "junk" 

# Pydantic will NOT throw an error here, even though "junk" is not a tuple!
c2 = BrokenCircle(radius=5)
print(c2.center) # Output: "junk"

```

*Note: This behavior exists for performance reasons—skipping validation on defaults speeds up model instantiation. You can force validation on defaults via `ConfigDict(validate_default=True)`.*

## 3. The Mutable Default "Gotcha" (And How Pydantic Fixes It)

In standard Python (like in function definitions or standard classes), assigning a **mutable object** (like a `list` or a `dict`) as a default value is a famous trap.

Because Python evaluates default arguments *once* at compile time, all instances would end up sharing the exact same list in memory. If one instance modifies the list, it modifies it for all future instances!

**Standard Python (The Bug):**

```python
def add_item(item, my_list=[]):
    my_list.append(item)
    return my_list

add_item("A") # Returns ["A"]
add_item("B") # Returns ["A", "B"] (Wait, where did "A" come from?!)

```

**Pydantic (The Fix):**
Pydantic is smart enough to handle this automatically. If you define a mutable default in a Pydantic model, Pydantic creates a **deep copy** of that object every time a new instance is created.

```python
class Student(BaseModel):
    name: str
    grades: list[int] = [] # In standard Python, this is a bug. In Pydantic, it's totally safe!

s1 = Student(name="Alice")
s1.grades.append(100)

s2 = Student(name="Bob")
print(s2.grades) # Output: [] (Bob starts with an empty list, Alice's data did not bleed over!)

```

*(Note: While Pydantic allows this, standard Python Dataclasses do not. In Dataclasses, you would be forced to use a `default_factory` to achieve this behavior).*

In [4]:
from pydantic import BaseModel, ValidationError

class Circle(BaseModel):
    center : tuple[int,int] = (0,0)
    radius : int

In [5]:
c1 =Circle(radius = 2)
print(c1.center, type(c1.center))
print(c1.radius)

(0, 0) <class 'tuple'>
2


In [7]:
c2 =Circle(center =(1,1), radius = 2)
print(c2.center, type(c2.center))
print(c2.radius)

(1, 1) <class 'tuple'>
2


In [9]:
try :
    Circle(center =(1,1))
except ValidationError as e:
    print(e)


1 validation error for Circle
radius
  Field required [type=missing, input_value={'center': (1, 1)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [ ]:
def add_item(item, my_list=["C"]):
    my_list.append(item)
    return my_list

print(add_item("A")) # Returns ["A"]
add_item("B") # Returns ["A", "B"] (Wait, where did "A" come from?!)

### 🧠 Pydantic Optional Fields & Defaults: Interview Prep

<details>
<summary><b>1. How do you make a field optional in a Pydantic model, and what happens when that field is omitted during deserialization?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You make a field optional simply by providing it with a default value in the model definition, exactly like providing a default argument in a standard Python function (e.g., `center: tuple[int, int] = (0, 0)`). 

When data is deserialized into the model, if the incoming payload is missing that specific field, Pydantic will automatically populate the model instance with the defined default value. It no longer raises a `ValidationError` for a missing required field.
</details>

<details>
<summary><b>2. If you assign a default value of the wrong type to a field (e.g., `radius: int = "junk"`), when does Pydantic catch this error by default?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
By default, Pydantic *does not* catch this error. 

Pydantic assumes that as the developer writing the class definition, you are providing logically sound defaults. It skips validation on the default values to save compute time and optimize performance. Therefore, if you omit the `radius` during instantiation, the model will successfully initialize with the string `"junk"` as the radius. (Though you can force Pydantic to validate defaults via `validate_default=True` in the model's Field configuration).
</details>

<details>
<summary><b>3. In standard Python, defining a mutable object (like a list or dict) as a default function argument is a dangerous anti-pattern. Why is this, and how does Pydantic handle mutable defaults differently in `BaseModel`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
In standard Python, default arguments are evaluated exactly *once* when the function (or class) is compiled. If you set a default to `my_list = []`, that exact single list object is stored in memory. Every time the function is called without that argument, it shares that *same* list. If you append to it, the list grows across all future function calls, leading to massive bugs.

In standard Python `dataclasses`, you fix this using `default_factory`. 

However, Pydantic goes a step further. If you define a mutable default directly in a Pydantic model (e.g., `tags: list[str] = []`), Pydantic intercepts this under the hood. Every time a new instance of the model is created, Pydantic automatically performs a **deep copy** of that mutable default. This ensures that every model instance gets its own isolated list, making it perfectly safe to write `tags: list = []` in Pydantic.
</details>

<details>
<summary><b>4. A Junior developer writes `tags: list[str] = None` to make a field optional. What is the semantic difference between that and writing `tags: list[str] = []` in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
While both make the field optional, they lead to entirely different downstream handling. 

If they use `= None`, the default state of the field is exactly `None`. Downstream code must constantly perform `if user.tags is not None:` checks before trying to iterate or append to the tags, otherwise, they will get a `TypeError`.

If they use `= []`, the default state is an empty list. Downstream code can safely assume `user.tags` is always an iterable list. You can safely call `for tag in user.tags:` or `user.tags.append("new")` without writing defensive `None` checks, leading to much cleaner code.
</details>

<details>
<summary><b>5. If you have an optional field but you *must* calculate the default value dynamically at runtime (e.g., setting a `created_at` timestamp), how do you do this in Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You cannot assign a dynamic function call directly to the attribute (e.g., `created_at: datetime = datetime.now()`). If you do, `datetime.now()` executes only once when the module is imported, meaning every single instance created during that application lifecycle will share the exact same timestamp.

Instead, you use Pydantic's `Field` with a `default_factory`. 
You define it as: `created_at: datetime = Field(default_factory=datetime.now)`. This tells Pydantic to execute the `datetime.now` callable fresh every single time a new instance is instantiated.
</details>
